# 05 · Instance Separation & Post-processing

**Goals**
- Turn the baseline's probability maps into better filament instances, without retraining
- Each idea targets an error type measured in notebook 04 and is kept only if it raises **PQ**
- Check the gain honestly on folds the settings were not tuned on, and measure the extra inference time

**Starting point (notebook 04, first run).** Instances = connected components of `p > threshold` on the disk,
components smaller than `min_area` dropped. Of the ground-truth filaments, 61% were matched, 19% **missed**,
10% **boundary** (one prediction, IoU ≤ 0.5), 5% **split** into several pieces and 4.5% **merged** with a
neighbour; PQ was still rising at the largest `min_area` tried, i.e. small spurious blobs cost a lot.

| Idea | Targets | How |
|---|---|---|
| Hysteresis threshold | split, boundary, spurious | keep a component of `p > low` only if it contains a confident pixel `p > high`: faint filament ends stay attached, faint blobs disappear |
| Gap grouping | split | pieces closer than `gap` px become one instance (the pixels themselves are not changed) |
| Boundary growth | boundary | grow every instance by `grow` px into the background, in case predictions are thinner than the annotated polygons |
| Test-time augmentation | all | average the probability over the 8 rotations/flips of the image (the disk is rotation invariant) |

**Inputs, all from previous notebooks:** the CSVs (01–02), the preprocessed images `cache/full_1024/` (02) and the
trained models + `baseline_runs.csv` (04). On Kaggle, attach the preprocessed-images dataset and the **output of
notebook 04** (*Add Input → Notebook Output Files*). `COS711_SMOKE=1` checks the pipeline on a few images on CPU.

In [ ]:
# --- Setup: works both locally and on Kaggle ---
import os, sys
from pathlib import Path

IN_KAGGLE = os.path.exists("/kaggle/input")

if IN_KAGGLE:
    # Pull the shared src/ code from GitHub (requires Internet ON in notebook settings)
    if not Path("/kaggle/working/COS711-A2").exists():
        !git clone -q https://github.com/AryanMohanlall/COS711-A2.git /kaggle/working/COS711-A2
    ROOT = Path("/kaggle/working/COS711-A2")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(ROOT))

from src.data import get_data_dir
DATA_DIR = get_data_dir()
print("Kaggle:", IN_KAGGLE, "| data dir:", DATA_DIR)

try:
    import segmentation_models_pytorch  # noqa: F401
except ImportError:
    %pip install -q segmentation-models-pytorch

In [ ]:
import json, time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm

from src.data import load_annotations
from src.preprocess import load_preprocessed, disk_mask
from src.augment import NO_AUG
from src.dataset import FilamentDataset
from src.model import build_model
from src.train import make_loader, predict, summarise
from src.postprocess import components, drop_small, upsample
from src.metrics import match_instances, pq_from_counts, error_types

SMOKE = os.environ.get("COS711_SMOKE") == "1"   # tiny CPU run: checks the pipeline, numbers meaningless
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUT_DIR = ROOT / "outputs" / "smoke" if SMOKE else ROOT
FIG_DIR = OUT_DIR / "figures" / "05"
ART_OUT = OUT_DIR / "artifacts"
for d in (FIG_DIR, ART_OUT):
    d.mkdir(parents=True, exist_ok=True)

def savefig(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.precision", 4)

S, FULL = 1024, 2048              # working / evaluation resolution (as notebook 04)
LIMB_CUT = 0.985
BATCH = 2 if SMOKE else 8
WORKERS = 0 if SMOKE else min(4, os.cpu_count())
VAL_FOLD = 0

def previous_output(rel):
    # A file written by an earlier notebook: an attached notebook output on Kaggle if there is one, else the repo copy
    hits = sorted(Path("/kaggle/input").glob(f"**/{rel}")) if IN_KAGGLE else []
    hits = [h for h in hits + [OUT_DIR / rel, ROOT / rel] if h.exists()]
    if not hits:
        raise FileNotFoundError(f"{rel} not found: run the earlier notebooks first"
                                + (" and attach their outputs (Add Input → Notebook Output Files)" if IN_KAGGLE else ""))
    print(f"{rel:40s} ← {hits[0]}")
    return hits[0]

def preprocessed_dir(probe):
    # cache/full_<S>/ from notebook 02: in the repo locally; on Kaggle, wherever an attached input holds the .npy files
    if IN_KAGGLE:
        hits = sorted(f.parent for f in Path("/kaggle/input").glob(f"**/{probe}"))
    else:
        hits = [ROOT / "cache" / f"full_{S}"]
    hits = [h for h in hits if (h / probe).exists()]
    if not hits:
        raise FileNotFoundError(f"preprocessed images ({probe}) not found: run notebook 02"
                                + (", upload cache/full_{S} as a Kaggle dataset and attach it" if IN_KAGGLE else ""))
    found = hits[0]
    if found.name != f"full_{S}":   # load_preprocessed reads <cache_dir>/full_<S>/<obs>.npy
        link = Path("/tmp/cos711_cache") / f"full_{S}"
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists():
            link.symlink_to(found, target_is_directory=True)
        found = link
    print(f"{f'cache/full_{S}':40s} ← {hits[0]}")
    return found.parent

IMAGE_STATS_CSV = previous_output("artifacts/image_stats.csv")            # notebook 01
EXCLUDED_CSV = previous_output("artifacts/excluded_annotations.csv")      # notebook 02
FOLDS_CSV = previous_output("artifacts/folds.csv")                        # notebook 02
CACHE_DIR = preprocessed_dir(f"{pd.read_csv(FOLDS_CSV).obs.iloc[0]}.npy")  # notebook 02
RUNS_CSV = previous_output("artifacts/baseline_runs.csv")                 # notebook 04

# Best fold-0 model of notebook 04 and its tuned post-processing
runs = pd.read_csv(RUNS_CSV).drop_duplicates("name", keep="last")
best = runs[(runs.fold == VAL_FOLD) & ~runs.name.str.contains("_fold")].sort_values("pq").iloc[-1]
BEST, T, A = best["name"], float(best.threshold), int(best.min_area)
CKPT = previous_output(f"checkpoints/{BEST}.pt")
print(f"\nbaseline: {BEST}  threshold {T}  min_area {A}  PQ {best.pq:.4f}")
print("device:", DEVICE, torch.cuda.get_device_name() if DEVICE.type == "cuda" else "", "| smoke:", SMOKE)

## 1. Validation predictions

The same data, normalisation and validation fold as notebook 04 (the normalisation is recomputed with the
same code and sample, so it is identical). Probabilities are predicted once, plain and with test-time
augmentation, and every post-processing variant below is scored on them.

In [ ]:
images, anns = load_annotations(DATA_DIR)
excluded = pd.read_csv(EXCLUDED_CSV)
anns = anns[~anns.id.isin(excluded.ann_id)].reset_index(drop=True)
folds = pd.read_csv(FOLDS_CSV)
geo_full = pd.read_csv(IMAGE_STATS_CSV).set_index("obs")[["cx", "cy", "radius"]]
paths = images.drop_duplicates("obs").set_index("obs").path

if SMOKE:   # a few observations per fold
    folds = folds.groupby("fold").head(3).reset_index(drop=True)

def split(fold):
    return folds.loc[folds.fold != fold, "obs"].tolist(), folds.loc[folds.fold == fold, "obs"].tolist()

# Input normalisation exactly as in notebook 04
train_obs, val_obs = split(VAL_FOLD)
probe = FilamentDataset(train_obs, images, anns, geo_full, size=S, cache_dir=CACHE_DIR)
v = []
for o in probe.obs[:: max(1, len(probe.obs) // 100)]:
    g = tuple(c * S / FULL for c in geo_full.loc[o])
    x = load_preprocessed(paths[o], tuple(geo_full.loc[o]), S, cache_dir=CACHE_DIR)
    v.append(x[disk_mask(x.shape, *g, LIMB_CUT)][::7])
v = np.concatenate(v)
med = float(np.median(v))
NORM = (med, float(1.4826 * np.median(np.abs(v - med))))
print("normalisation (centre, scale):", tuple(round(c, 4) for c in NORM))

def val_set(fold):
    _, va_obs = split(fold)
    va = FilamentDataset(va_obs, images, anns, geo_full, size=S, norm=NORM, aug=NO_AUG, cache_dir=CACHE_DIR)
    return va, make_loader(va, BATCH, train=False, workers=WORKERS)

def load_model(path):
    m = build_model(weights=None).to(DEVICE)
    m.load_state_dict(torch.load(path, map_location=DEVICE))
    return m.eval()

@torch.no_grad()
def predict_tta(model, loader):
    """{obs: probability averaged over the 8 rotations / flips} (float16, working resolution)."""
    out = {}
    for x, _, idx in tqdm(loader, desc="TTA", leave=False):
        x, acc = x.to(DEVICE), 0
        for k in range(4):
            for flip in (False, True):
                t = torch.rot90(x, k, (2, 3))
                t = t.flip(3) if flip else t
                with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                    p = torch.sigmoid(model(t).float())
                p = p.flip(3) if flip else p
                acc = acc + torch.rot90(p, -k, (2, 3))
        for i, p in zip(idx.tolist(), (acc / 8)[:, 0].cpu().numpy()):
            out[loader.dataset.obs[i]] = p.astype(np.float16)
    return out

model = load_model(CKPT)
va, va_dl = val_set(VAL_FOLD)
gt_fn = lambda o: va.masks(o, FULL)
disks = {o: va.geo[o] for o in va.obs}

t0 = time.time(); probs = predict(model, va_dl); t_plain = time.time() - t0
t0 = time.time(); probs_tta = predict_tta(model, va_dl); t_tta = time.time() - t0
print(f"{len(probs)} validation images | network: plain {1000 * t_plain / len(probs):.0f} ms/image, "
      f"TTA {1000 * t_tta / len(probs):.0f} ms/image")

## 2. Post-processing pipeline and evaluation

`instances(p, …)` turns a full-resolution probability map into an instance map; with only `high` and
`min_area` set it is exactly notebook 04's post-processing. Every variant is scored with the notebook 04
protocol: PQ at 2048 px against every annotation set of every validation observation, counts pooled.

In [ ]:
def disk_kernel(r):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))

def instances(p, on_disk, high, low=None, gap=0, min_area=0, grow=0):
    """p: probability at full resolution; on_disk: boolean disk mask. Returns an instance map (0 = background).
    high: threshold (or the seed threshold with hysteresis); low: hysteresis threshold for the extent;
    gap: pieces closer than this (px) form one instance; min_area: drop smaller instances (px);
    grow: grow instances by this many px into the background."""
    fg = (p > (high if low is None else low)) & on_disk
    lab, _ = components(fg)
    if low is not None:   # hysteresis: keep components of p > low that contain a pixel p > high
        keep = np.zeros(lab.max() + 1, bool)
        keep[np.unique(lab[fg & (p > high)])] = True
        keep[0] = False
        fg = keep[lab]
    if gap:   # group pieces whose dilations touch; pixels stay the predicted ones
        groups, _ = components(cv2.dilate(fg.astype(np.uint8), disk_kernel(gap)) > 0)
        lab = np.where(fg, groups, 0)
    elif low is not None:
        lab, _ = components(fg)
    lab = drop_small(lab, np.bincount(lab.ravel()), min_area)
    if grow:
        grown = cv2.dilate(lab.astype(np.uint16), disk_kernel(grow)).astype(lab.dtype)
        lab = np.where((lab == 0) & on_disk, grown, lab)
    return lab

def evaluate_configs(probs, configs, gt_fn, disks):
    """Per-image TP / FP / FN / IoU sums for every named config ({name: instances kwargs})."""
    rows = []
    for o, prob in tqdm(probs.items(), desc=f"{len(configs)} configs", leave=False):
        gts = gt_fn(o)
        p = upsample(prob, FULL)
        on_disk = disk_mask(p.shape, *disks[o], LIMB_CUT)
        for name, kw in configs.items():
            pred = instances(p, on_disk, **kw)
            c = np.zeros(4)
            for gt in gts:
                m = match_instances(gt, pred)
                c += (m["tp"], m["fp"], m["fn"], m["iou_sum"])
            rows.append({"obs": o, "config": name, "tp": c[0], "fp": c[1], "fn": c[2], "iou_sum": c[3]})
    return pd.DataFrame(rows)

def score(probs, configs, gt_fn=gt_fn, disks=disks):
    """Pooled PQ / SQ / RQ per config, best first, with the config's settings as columns."""
    res = summarise(evaluate_configs(probs, configs, gt_fn, disks), by=["config"])
    res = res.join(pd.DataFrame.from_dict(configs, orient="index"))
    return res.sort_values("pq", ascending=False)

def name(kw):
    return " ".join(f"{k}={v}" for k, v in kw.items() if v is not None and v != 0) or "none"

baseline = dict(high=T, min_area=A)
base_res = score(probs, {"baseline": baseline})
print(f"baseline reproduced: PQ {base_res.pq.iloc[0]:.4f} (notebook 04: {best.pq:.4f})")
base_res[["pq", "sq", "rq", "tp", "fp", "fn"]]

## 3. Hysteresis thresholding

`high` decides *whether* there is a filament, `low` how far it extends. A single threshold has to do both:
high enough to suppress faint false blobs, which then cuts faint filament ends off or splits the filament.
`min_area` is retuned together, since hysteresis already removes many small blobs.

In [ ]:
HIGHS = (0.6, 0.7, 0.8, 0.9)
LOWS = (None, 0.3, 0.4, 0.5)
MIN_AREAS = (200, 400, 800, 1600)

hyst_configs = {}
for h in HIGHS:
    for l in LOWS:
        if l is None or l < h:
            for a in MIN_AREAS:
                kw = dict(high=h, low=l, min_area=a)
                hyst_configs[name(kw)] = kw
hyst = score(probs, hyst_configs)
best_hyst = hyst_configs[hyst.index[0]]
print(f"best: {hyst.index[0]}  PQ {hyst.pq.iloc[0]:.4f}  (baseline {base_res.pq.iloc[0]:.4f})")

# PQ for each (high, low), at the best min_area for that pair
grid = (hyst.assign(low=hyst.low.fillna(0)).groupby(["high", "low"]).pq.max().unstack("low")
        .rename(columns={0: "none"}))
fig, ax = plt.subplots(figsize=(6.5, 3.5))
im = ax.imshow(grid.values, cmap="viridis", aspect="auto")
ax.set(xticks=range(grid.shape[1]), xticklabels=grid.columns, yticks=range(grid.shape[0]), yticklabels=grid.index,
       xlabel="low (hysteresis)", ylabel="high", title="Validation PQ (best min_area per cell)")
for (i, j), val in np.ndenumerate(grid.values):
    if not np.isnan(val):
        ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=7, color="w")
plt.colorbar(im); plt.tight_layout(); savefig("hysteresis_grid"); plt.show()
hyst.head(10)[["pq", "sq", "rq", "tp", "fp", "fn"]]

## 4. Gap grouping

Filaments are often broken into pieces along their length. Pieces whose `gap`-px dilations touch become one
instance; their pixels are not changed, so boundaries (SQ) are unaffected. The risk is merging genuinely
separate neighbours (only 2.9% have a neighbour within 10 px, notebook 01). `min_area` is retuned with it,
since grouped instances are larger.

In [ ]:
GAPS = (0, 3, 6, 10, 15, 25)
a0 = best_hyst["min_area"]
gap_configs = {}
for g in GAPS:
    for a in sorted({a0 // 2, a0, a0 * 2}):
        kw = {**best_hyst, "gap": g, "min_area": a}
        gap_configs[name(kw)] = kw
gaps = score(probs, gap_configs)
best_gap = gap_configs[gaps.index[0]]
print(f"best: {gaps.index[0]}  PQ {gaps.pq.iloc[0]:.4f}")

fig, ax = plt.subplots(figsize=(6, 3))
per_gap = gaps.sort_values("pq").groupby("gap").tail(1).set_index("gap").sort_index()[["pq", "sq", "rq", "min_area"]]
per_gap[["pq", "sq", "rq"]].plot(ax=ax, marker="o")
ax.set(xlabel="gap (full-res px)", ylabel="validation score", title="Gap grouping (best min_area per gap)")
plt.tight_layout(); savefig("gap_grouping"); plt.show()
per_gap

## 5. Boundary growth

If predicted filaments are systematically thinner than the annotated polygons, boundary errors (one
prediction, IoU ≤ 0.5) can be turned into matches by growing every instance by a few pixels.

In [ ]:
GROWS = (0, 1, 2, 3, 5, 8)
grow_configs = {name({**best_gap, "grow": g}): {**best_gap, "grow": g} for g in GROWS}
grows = score(probs, grow_configs)
best_grow = grow_configs[grows.index[0]]
print(f"best: {grows.index[0]}  PQ {grows.pq.iloc[0]:.4f}")
grows.sort_values("grow")[["grow", "pq", "sq", "rq", "tp", "fp", "fn"]]

## 6. Test-time augmentation

The model is applied to the 8 rotations / flips of each image and the probabilities are averaged. The disk is
rotation invariant, so every view is a valid image. Averaging changes the probability scale, so the thresholds
are retuned around the chosen pipeline.

In [ ]:
tta_configs = {}
for dh in (-0.1, 0.0, 0.1):
    h = round(min(0.95, max(0.1, best_grow["high"] + dh)), 2)
    if best_grow["low"] is None or best_grow["low"] < h:
        kw = {**best_grow, "high": h}
        tta_configs[name(kw)] = kw
tta = score(probs_tta, tta_configs)
plain_final = grows.pq.iloc[0]
use_tta = tta.pq.iloc[0] > plain_final
FINAL = tta_configs[tta.index[0]] if use_tta else best_grow
print(f"plain {plain_final:.4f} vs TTA {tta.pq.iloc[0]:.4f} ({tta.index[0]}) → use TTA: {use_tta}")
tta[["pq", "sq", "rq", "tp", "fp", "fn"]]

## 7. What changed: error types, baseline vs final

In [ ]:
final_probs = probs_tta if use_tta else probs

def error_breakdown(probs, kw):
    err = []
    for o, prob in tqdm(probs.items(), desc="error types", leave=False):
        p = upsample(prob, FULL)
        pred = instances(p, disk_mask(p.shape, *disks[o], LIMB_CUT), **kw)
        for gt in gt_fn(o):
            err.append(error_types(gt, pred))
    tot = pd.DataFrame(err).sum()
    gt_total = tot[["matched", "missed", "merged", "split", "boundary"]].sum()
    return pd.concat([100 * tot[["matched", "missed", "merged", "split", "boundary"]] / gt_total,
                      pd.Series({"spurious (count)": tot.spurious})])

errs = pd.DataFrame({"baseline": error_breakdown(probs, baseline), "final": error_breakdown(final_probs, FINAL)})
fig, ax = plt.subplots(figsize=(7, 3))
errs.drop("spurious (count)").plot.bar(ax=ax)
ax.set(ylabel="% of GT filaments", title="Fate of ground-truth filaments (validation)"); plt.xticks(rotation=0)
plt.tight_layout(); savefig("error_types_before_after"); plt.show()
errs.round(1)

## 8. Honest check on folds 1–4

Every setting above was chosen on fold 0, so the fold-0 gain is optimistic. Notebook 04's cross-validation
models (`<best>_fold1…4`) were never used for tuning: the baseline and the final pipeline are applied to each
with **fixed** settings, and the gain is reported as mean ± std over the folds. Skipped if notebook 04 was run
without cross-validation.

In [ ]:
cv_rows = []
for f in sorted(folds.fold.unique()):
    if f == VAL_FOLD:
        continue
    try:
        ckpt = previous_output(f"checkpoints/{BEST}_fold{f}.pt")
    except FileNotFoundError:
        print(f"fold {f}: no checkpoint, skipped")
        continue
    m = load_model(ckpt)
    va_f, dl_f = val_set(f)
    gt_f = lambda o, va_f=va_f: va_f.masks(o, FULL)
    disks_f = {o: va_f.geo[o] for o in va_f.obs}
    pr = predict_tta(m, dl_f) if use_tta else predict(m, dl_f)
    pr_plain = predict(m, dl_f) if use_tta else pr
    b = score(pr_plain, {"baseline": baseline}, gt_f, disks_f).pq.iloc[0]
    fi = score(pr, {"final": FINAL}, gt_f, disks_f).pq.iloc[0]
    cv_rows.append({"fold": f, "baseline": b, "final": fi, "gain": fi - b})
    del m; torch.cuda.empty_cache()

cv = pd.DataFrame(cv_rows).set_index("fold") if cv_rows else None
if cv is not None:
    print(f"folds {', '.join(map(str, cv.index))}: baseline {cv.baseline.mean():.3f} ± {cv.baseline.std():.3f} → "
          f"final {cv.final.mean():.3f} ± {cv.final.std():.3f}  (gain {cv.gain.mean():+.3f} ± {cv.gain.std():.3f})")
    display(cv)

## 9. Cost: post-processing and inference time

Post-processing time per image at full resolution, and the network time with and without TTA (section 1).

In [ ]:
def postprocess_ms(probs, kw, n=20):
    ts = []
    for o in list(probs)[:n]:
        t0 = time.perf_counter()
        p = upsample(probs[o], FULL)
        instances(p, disk_mask(p.shape, *disks[o], LIMB_CUT), **kw)
        ts.append(time.perf_counter() - t0)
    return 1000 * float(np.mean(ts))

cost = pd.DataFrame({
    "network ms / image": [1000 * t_plain / len(probs), 1000 * (t_tta if use_tta else t_plain) / len(probs)],
    "post-processing ms / image": [postprocess_ms(probs, baseline), postprocess_ms(final_probs, FINAL)],
}, index=["baseline", "final"])
cost["total ms / image"] = cost.sum(axis=1)
cost.round(1)

## Summary: Panoptic Quality

Each step is added on top of the previous one (fold 0; settings tuned there). The final settings are saved to
`artifacts/postprocess.json` for the inference notebook.

In [ ]:
steps = pd.DataFrame({
    "PQ (fold 0)": [base_res.pq.iloc[0], hyst.pq.iloc[0], gaps.pq.iloc[0], grows.pq.iloc[0], tta.pq.iloc[0]],
    "settings": [name(baseline), hyst.index[0], gaps.index[0], grows.index[0], tta.index[0]],
}, index=["baseline (notebook 04)", "+ hysteresis", "+ gap grouping", "+ boundary growth", "+ TTA"])
display(steps)

print(f"Final: {name(FINAL)}{' + TTA' if use_tta else ''}, PQ {max(plain_final, tta.pq.iloc[0]):.4f} on fold 0 "
      f"(baseline {base_res.pq.iloc[0]:.4f})")
if cv is not None:
    print(f"Folds {', '.join(map(str, cv.index))} (not tuned on): baseline {cv.baseline.mean():.3f} ± {cv.baseline.std():.3f}"
          f" → final {cv.final.mean():.3f} ± {cv.final.std():.3f}")

with open(ART_OUT / "postprocess.json", "w") as fh:
    json.dump({"model": BEST, "tta": bool(use_tta), **FINAL}, fh, indent=1)